# 第39章：正则化线性模型

本读者Notebook只回放fixtures/late中的已接受历史构造摘要，不重训、不重演500次模拟、不重建缺失逐行预测。Ridge目标为 $\|y-Xw\|^2+\alpha\|w\|^2$，Lasso为 $\|y-Xw\|^2/(2n)+\alpha\|w\|_1$，Elastic Net的惩罚为 $\alpha\rho\|w\|_1+\alpha(1-\rho)\|w\|^2/2$，alpha缩放不能混用。

手算、偏差方差、共线性与六模型表均为保存结果；显示两张旧图。创建RUN_DIR但不新写结果。原A8-R1候选文字是历史记录，当前接受身份沿根收据；共同日期/配对均值仍未复算，旧t不作推断展示。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch39_regularized_models_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import json
import numpy as np, pandas as pd
from IPython.display import display, Image
CANDIDATE = ROOT / 'fixtures/late'
FROZEN = json.loads((CANDIDATE / 'ch39/ch39_results.json').read_text())
AUDIT = json.loads((CANDIDATE / 'display_audit.json').read_text())
print('Accepted historical summary display; no training or Ch39 row reconstruction.')


## 1. 手算：$X^\top X=4I$，$z=(2,1)$

In [ ]:
display(pd.DataFrame(FROZEN['hand_example']['closed_form'], index=['w1','w2']).T)
print('Frozen max difference:', FROZEN['hand_example']['max_abs_diff'])

## 2. 偏差—方差：$\alpha^\ast=\sigma^2/w^2$

In [ ]:
display(pd.DataFrame({k: FROZEN['bias_variance'][k] for k in ['alphas','mse_theory','mse_sim']}))
display(Image(filename=str(CANDIDATE/'ch39/fig39_2_bias_variance.png')))
print('Historical simulations only, not rerun.')

## 3. 共线性（相关 0.99，500 次重复）

In [ ]:
display(pd.DataFrame(FROZEN['collinearity']['results']).T)
print('Frozen500 repeats, not rerun.')

## 4. 冻结六模型展示回放

IC是各自有效日均值；Ridge750/Lasso441/EN624，候选标签日750。共同日及其配对均值/逐日缺失原因未能复算，不得用P06另一个实验替代。旧压缩NW(4)不是四交易日HAC，t不报告；该Notebook不调用panel_case。

In [ ]:
rows={}
for model, v in FROZEN['panel_case']['results'].items():
 p=v['prediction']; n=v['portfolio']['fee_0.001']
 rows[model]={'own_IC': p['ic_mean'], 'own_valid_dates':p.get('ic_dates',0), 'candidate_dates':p.get('n_dates',750), 'paired_dates':'not_evaluated', 'paired_mean':'not_evaluated', 't_display':'not_reported', 'net_Sharpe':n['sharpe'], 'net_total':n['total_return']}
display(pd.DataFrame(rows).T)
display(Image(filename=str(CANDIDATE/'ch39/fig39_3_display_revision.png')))
assert AUDIT['chapter39']['lasso']['paired_ridge_dates'] is None

### 39-C03完整答案
有效集D_m与Ridge交集才定义配对比较。无冻结行则未能复算，不补造。0/2/3/5/8的压缩第4观察滞后跨8交易日，因此旧NW(4)不能解释成4交易日HAC。